# FF5ファクターモデル：現時点の相場での有効性分析

**作成日**: 2026-03-15
**目的**: Fama-French 5ファクター + モメンタムの現時点での有効性を評価

## 分析概要
1. FF5ファクターデータ読み込み
2. 基本統計（平均、標準偏差、シャープレシオ、t統計量）
3. 期間別分析（全期間、直近12ヶ月、6ヶ月、3ヶ月）
4. ローリング分析（12ヶ月窓）
5. レジーム分析（強気/弱気）
6. 現時点での有効ファクターランキング

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats

# 日本語フォント設定
plt.rcParams['font.family'] = 'MS Gothic'
plt.rcParams['axes.unicode_minus'] = False

# プロジェクトルート
PROJECT_ROOT = Path(r"C:\Users\yongr\claude project\workspace")

# データパス
FF5_PATH = PROJECT_ROOT / "legacy/_inbox/merged_data_all_stocks/factors/ff5_mom_factors_monthly.parquet"
SNAPSHOT_PATH = PROJECT_ROOT / "legacy/_inbox/merged_data_all_stocks/factors/month_end_snapshot.parquet"

# 出力ディレクトリ
OUTPUT_DIR = PROJECT_ROOT / "analyses/20260315_1500_ff5_current_effectiveness"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"プロジェクトルート: {PROJECT_ROOT}")
print(f"出力ディレクトリ: {OUTPUT_DIR}")

## 1. データ読み込み

In [ ]:
# FF5ファクターデータ読み込み
df_factors = pd.read_parquet(FF5_PATH, engine="pyarrow")

# 列名確認
print("列名:", df_factors.columns.tolist())
print(f"\nデータ期間: {df_factors['MonthEnd'].min()} ~ {df_factors['MonthEnd'].max()}")
print(f"総月数: {len(df_factors)}")

# 先頭・末尾表示
display(df_factors.head())
display(df_factors.tail())

In [ ]:
# 日付型に変換
df_factors['MonthEnd'] = pd.to_datetime(df_factors['MonthEnd'])
df_factors = df_factors.sort_values('MonthEnd').reset_index(drop=True)

# ファクター列を抽出
factor_cols = ['MKT', 'SMB', 'HML', 'RMW', 'CMA', 'WML']

# 欠損値確認
print("欠損値:")
print(df_factors[factor_cols].isnull().sum())

# 基本統計
print("\n基本統計:")
display(df_factors[factor_cols].describe())

## 2. 全期間の基本統計分析

In [ ]:
def calculate_performance_stats(returns_df, factor_cols, period_name="全期間"):
    """
    ファクターリターンのパフォーマンス統計を計算
    
    Parameters:
    -----------
    returns_df : DataFrame
        ファクターリターンのデータフレーム
    factor_cols : list
        ファクター列のリスト
    period_name : str
        期間名（表示用）
    
    Returns:
    --------
    stats_df : DataFrame
        統計サマリー
    """
    n_months = len(returns_df)
    
    stats = {}
    for col in factor_cols:
        data = returns_df[col].dropna()
        
        mean_ret = data.mean()
        std_ret = data.std()
        
        # 年率換算（月次データ）
        annual_mean = mean_ret * 12
        annual_std = std_ret * np.sqrt(12)
        
        # シャープレシオ（リスクフリーレート = 0と仮定）
        sharpe = mean_ret / std_ret if std_ret > 0 else 0
        annual_sharpe = sharpe * np.sqrt(12)
        
        # t統計量
        t_stat = (mean_ret / std_ret) * np.sqrt(len(data)) if std_ret > 0 else 0
        
        # p値（両側検定）
        p_value = 2 * (1 - stats.t.cdf(abs(t_stat), df=len(data)-1))
        
        # 累積リターン
        cumulative_ret = (1 + data).prod() - 1
        
        stats[col] = {
            '月次平均': mean_ret,
            '月次標準偏差': std_ret,
            '年率リターン': annual_mean,
            '年率ボラティリティ': annual_std,
            'シャープレシオ（年率）': annual_sharpe,
            't統計量': t_stat,
            'p値': p_value,
            '累積リターン': cumulative_ret,
            '観測月数': len(data)
        }
    
    stats_df = pd.DataFrame(stats).T
    stats_df.index.name = 'ファクター'
    
    # 有意性フラグ（p < 0.05）
    stats_df['有意'] = stats_df['p値'] < 0.05
    
    print(f"\n{'='*60}")
    print(f"{period_name}のパフォーマンス統計")
    print(f"{'='*60}")
    
    return stats_df

# 全期間の統計
stats_all = calculate_performance_stats(df_factors, factor_cols, "全期間")
display(stats_all.sort_values('シャープレシオ（年率）', ascending=False))

## 3. 期間別分析（直近12ヶ月、6ヶ月、3ヶ月）

In [ ]:
# 最新月
latest_month = df_factors['MonthEnd'].max()
print(f"最新月: {latest_month}")

# 期間別データ抽出
def get_recent_data(df, months_back):
    cutoff_date = latest_month - pd.DateOffset(months=months_back)
    return df[df['MonthEnd'] > cutoff_date].copy()

df_12m = get_recent_data(df_factors, 12)
df_6m = get_recent_data(df_factors, 6)
df_3m = get_recent_data(df_factors, 3)

print(f"\n直近12ヶ月: {len(df_12m)}ヶ月")
print(f"直近6ヶ月: {len(df_6m)}ヶ月")
print(f"直近3ヶ月: {len(df_3m)}ヶ月")

In [ ]:
# 各期間の統計計算
stats_12m = calculate_performance_stats(df_12m, factor_cols, "直近12ヶ月")
display(stats_12m.sort_values('シャープレシオ（年率）', ascending=False))

In [ ]:
stats_6m = calculate_performance_stats(df_6m, factor_cols, "直近6ヶ月")
display(stats_6m.sort_values('シャープレシオ（年率）', ascending=False))

In [ ]:
stats_3m = calculate_performance_stats(df_3m, factor_cols, "直近3ヶ月")
display(stats_3m.sort_values('シャープレシオ（年率）', ascending=False))

## 4. 期間別比較（年率リターン）

In [ ]:
# 期間別比較表作成
comparison = pd.DataFrame({
    '全期間': stats_all['年率リターン'],
    '直近12ヶ月': stats_12m['年率リターン'],
    '直近6ヶ月': stats_6m['年率リターン'],
    '直近3ヶ月': stats_3m['年率リターン']
})

print("\n年率リターン比較（期間別）:")
display(comparison.sort_values('直近12ヶ月', ascending=False))

# CSVに保存
comparison.to_csv(OUTPUT_DIR / "factor_performance_by_period.csv")
print(f"\n保存完了: {OUTPUT_DIR / 'factor_performance_by_period.csv'}")

In [ ]:
# 可視化
fig, ax = plt.subplots(figsize=(12, 6))
comparison.T.plot(kind='bar', ax=ax)
ax.set_title('FF5ファクター年率リターン（期間別）', fontsize=14, fontweight='bold')
ax.set_xlabel('期間', fontsize=12)
ax.set_ylabel('年率リターン', fontsize=12)
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.8)
ax.legend(title='ファクター', bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "factor_returns_by_period.png", dpi=150)
plt.show()

print(f"保存完了: {OUTPUT_DIR / 'factor_returns_by_period.png'}")

## 5. シャープレシオ比較

In [ ]:
# シャープレシオ比較表
sharpe_comparison = pd.DataFrame({
    '全期間': stats_all['シャープレシオ（年率）'],
    '直近12ヶ月': stats_12m['シャープレシオ（年率）'],
    '直近6ヶ月': stats_6m['シャープレシオ（年率）'],
    '直近3ヶ月': stats_3m['シャープレシオ（年率）']
})

print("\nシャープレシオ比較（期間別）:")
display(sharpe_comparison.sort_values('直近12ヶ月', ascending=False))

# CSVに保存
sharpe_comparison.to_csv(OUTPUT_DIR / "factor_sharpe_by_period.csv")
print(f"\n保存完了: {OUTPUT_DIR / 'factor_sharpe_by_period.csv'}")

In [ ]:
# 可視化
fig, ax = plt.subplots(figsize=(12, 6))
sharpe_comparison.T.plot(kind='bar', ax=ax)
ax.set_title('FF5ファクターシャープレシオ（期間別）', fontsize=14, fontweight='bold')
ax.set_xlabel('期間', fontsize=12)
ax.set_ylabel('シャープレシオ（年率）', fontsize=12)
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.8)
ax.legend(title='ファクター', bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "factor_sharpe_by_period.png", dpi=150)
plt.show()

print(f"保存完了: {OUTPUT_DIR / 'factor_sharpe_by_period.png'}")

## 6. 累積リターンの推移

In [ ]:
# 累積リターン計算
cumulative_returns = pd.DataFrame(index=df_factors['MonthEnd'])

for col in factor_cols:
    cumulative_returns[col] = (1 + df_factors[col]).cumprod() - 1

# 最終累積リターン表示
print("\n最終累積リターン:")
final_returns = cumulative_returns.iloc[-1].sort_values(ascending=False)
print(final_returns)

# 可視化
fig, ax = plt.subplots(figsize=(14, 7))
for col in factor_cols:
    ax.plot(cumulative_returns.index, cumulative_returns[col], label=col, linewidth=2)

ax.set_title('FF5ファクター累積リターン推移', fontsize=14, fontweight='bold')
ax.set_xlabel('月末日', fontsize=12)
ax.set_ylabel('累積リターン', fontsize=12)
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.8)
ax.legend(loc='best', fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "factor_cumulative_returns.png", dpi=150)
plt.show()

print(f"\n保存完了: {OUTPUT_DIR / 'factor_cumulative_returns.png'}")

## 7. ローリング分析（12ヶ月窓）

In [ ]:
# ローリング平均リターン（12ヶ月窓）
window = 12

rolling_mean = df_factors[factor_cols].rolling(window=window).mean() * 12  # 年率換算
rolling_std = df_factors[factor_cols].rolling(window=window).std() * np.sqrt(12)  # 年率換算
rolling_sharpe = rolling_mean / rolling_std

# 日付インデックスを追加
rolling_mean.index = df_factors['MonthEnd']
rolling_std.index = df_factors['MonthEnd']
rolling_sharpe.index = df_factors['MonthEnd']

# 最新値表示
print("\n最新のローリングシャープレシオ（12ヶ月窓）:")
print(rolling_sharpe.iloc[-1].sort_values(ascending=False))

In [ ]:
# ローリング平均リターン可視化
fig, ax = plt.subplots(figsize=(14, 7))
for col in factor_cols:
    ax.plot(rolling_mean.index, rolling_mean[col], label=col, linewidth=2)

ax.set_title('FF5ファクターローリング平均リターン（12ヶ月窓、年率）', fontsize=14, fontweight='bold')
ax.set_xlabel('月末日', fontsize=12)
ax.set_ylabel('年率リターン', fontsize=12)
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.8)
ax.legend(loc='best', fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "factor_rolling_mean.png", dpi=150)
plt.show()

print(f"保存完了: {OUTPUT_DIR / 'factor_rolling_mean.png'}")

In [ ]:
# ローリングシャープレシオ可視化
fig, ax = plt.subplots(figsize=(14, 7))
for col in factor_cols:
    ax.plot(rolling_sharpe.index, rolling_sharpe[col], label=col, linewidth=2)

ax.set_title('FF5ファクターローリングシャープレシオ（12ヶ月窓、年率）', fontsize=14, fontweight='bold')
ax.set_xlabel('月末日', fontsize=12)
ax.set_ylabel('シャープレシオ（年率）', fontsize=12)
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.8)
ax.legend(loc='best', fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "factor_rolling_sharpe.png", dpi=150)
plt.show()

print(f"保存完了: {OUTPUT_DIR / 'factor_rolling_sharpe.png'}")

## 8. ファクター間相関分析

In [ ]:
# 全期間の相関行列
corr_all = df_factors[factor_cols].corr()

print("\n全期間のファクター間相関:")
display(corr_all)

# ヒートマップ
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_all, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=1, cbar_kws={"shrink": 0.8}, ax=ax)
ax.set_title('FF5ファクター間相関（全期間）', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "factor_correlation_all.png", dpi=150)
plt.show()

print(f"保存完了: {OUTPUT_DIR / 'factor_correlation_all.png'}")

In [ ]:
# 直近12ヶ月の相関行列
corr_12m = df_12m[factor_cols].corr()

print("\n直近12ヶ月のファクター間相関:")
display(corr_12m)

# ヒートマップ
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_12m, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=1, cbar_kws={"shrink": 0.8}, ax=ax)
ax.set_title('FF5ファクター間相関（直近12ヶ月）', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "factor_correlation_12m.png", dpi=150)
plt.show()

print(f"保存完了: {OUTPUT_DIR / 'factor_correlation_12m.png'}")

## 9. レジーム分析（強気/弱気）

In [ ]:
# 市場環境の分類（MKTリターンで判定）
df_factors['Regime'] = df_factors['MKT'].apply(
    lambda x: 'Bull' if x > 0 else 'Bear'
)

print("\nレジーム分布:")
print(df_factors['Regime'].value_counts())

# レジーム別ファクターパフォーマンス
regime_performance = df_factors.groupby('Regime')[factor_cols].agg(['mean', 'std', 'count'])

print("\nレジーム別平均リターン（月次）:")
display(regime_performance.xs('mean', axis=1, level=1).T)

In [ ]:
# レジーム別シャープレシオ
regime_sharpe = {}
for regime in ['Bull', 'Bear']:
    df_regime = df_factors[df_factors['Regime'] == regime]
    sharpe = {}
    for col in factor_cols:
        mean_ret = df_regime[col].mean()
        std_ret = df_regime[col].std()
        sharpe[col] = (mean_ret / std_ret) * np.sqrt(12) if std_ret > 0 else 0
    regime_sharpe[regime] = sharpe

regime_sharpe_df = pd.DataFrame(regime_sharpe).T

print("\nレジーム別シャープレシオ（年率）:")
display(regime_sharpe_df)

# CSVに保存
regime_sharpe_df.to_csv(OUTPUT_DIR / "factor_performance_by_regime.csv")
print(f"\n保存完了: {OUTPUT_DIR / 'factor_performance_by_regime.csv'}")

In [ ]:
# レジーム別可視化
fig, ax = plt.subplots(figsize=(12, 6))
regime_sharpe_df.T.plot(kind='bar', ax=ax)
ax.set_title('FF5ファクターシャープレシオ（レジーム別）', fontsize=14, fontweight='bold')
ax.set_xlabel('ファクター', fontsize=12)
ax.set_ylabel('シャープレシオ（年率）', fontsize=12)
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.8)
ax.legend(title='レジーム', loc='best')
ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "factor_sharpe_by_regime.png", dpi=150)
plt.show()

print(f"保存完了: {OUTPUT_DIR / 'factor_sharpe_by_regime.png'}")

## 10. 最終サマリー：現時点での有効ファクターランキング

In [ ]:
# 総合ランキング作成
ranking = pd.DataFrame({
    '全期間_年率リターン': stats_all['年率リターン'],
    '全期間_シャープ': stats_all['シャープレシオ（年率）'],
    '全期間_t統計量': stats_all['t統計量'],
    '全期間_有意': stats_all['有意'],
    '12M_年率リターン': stats_12m['年率リターン'],
    '12M_シャープ': stats_12m['シャープレシオ（年率）'],
    '6M_年率リターン': stats_6m['年率リターン'],
    '6M_シャープ': stats_6m['シャープレシオ（年率）'],
    '3M_年率リターン': stats_3m['年率リターン'],
    '3M_シャープ': stats_3m['シャープレシオ（年率）']
})

# シャープレシオでソート（直近12ヶ月）
ranking_sorted = ranking.sort_values('12M_シャープ', ascending=False)

print("\n【最終ランキング】現時点で有効なファクター（直近12ヶ月シャープレシオ順）:")
print("="*80)
display(ranking_sorted)

# CSVに保存
ranking_sorted.to_csv(OUTPUT_DIR / "factor_final_ranking.csv")
print(f"\n保存完了: {OUTPUT_DIR / 'factor_final_ranking.csv'}")

## 11. 結論・推奨

In [ ]:
# Top3ファクター（直近12ヶ月シャープレシオ）
top3_factors = ranking_sorted.head(3).index.tolist()

print("\n【結論】現時点（2026-03時点、最新データ2025-12）で有効なファクター:")
print("="*80)
print(f"\nTop 3（直近12ヶ月シャープレシオ順）:")
for i, factor in enumerate(top3_factors, 1):
    ret_12m = ranking_sorted.loc[factor, '12M_年率リターン']
    sharpe_12m = ranking_sorted.loc[factor, '12M_シャープ']
    print(f"{i}. {factor}: 年率リターン {ret_12m:.2%}, シャープレシオ {sharpe_12m:.2f}")

print("\n【推奨戦略】")
print(f"1. 主力ファクター: {top3_factors[0]}")
print(f"2. 補助ファクター: {top3_factors[1]}, {top3_factors[2]}")
print(f"3. レジーム: 最新月は{'Bull（強気）' if df_factors.iloc[-1]['Regime'] == 'Bull' else 'Bear（弱気）'}")
print(f"4. ポートフォリオ構築: 上記ファクターを組み合わせたマルチファクター戦略を推奨")

print("\n【注意事項】")
print("- 最新データは2025-12-31（約3ヶ月前）")
print("- 2026年1月-3月のデータは未反映のため、直近の市場変動は考慮されていない")
print("- 実運用前に最新データで再検証を推奨")

## 完了

In [ ]:
print("\n" + "="*80)
print("分析完了")
print("="*80)
print(f"\n出力ディレクトリ: {OUTPUT_DIR}")
print("\n生成ファイル:")
print("- factor_performance_by_period.csv")
print("- factor_sharpe_by_period.csv")
print("- factor_performance_by_regime.csv")
print("- factor_final_ranking.csv")
print("- factor_returns_by_period.png")
print("- factor_sharpe_by_period.png")
print("- factor_cumulative_returns.png")
print("- factor_rolling_mean.png")
print("- factor_rolling_sharpe.png")
print("- factor_correlation_all.png")
print("- factor_correlation_12m.png")
print("- factor_sharpe_by_regime.png")